# Phase 6 — Publication Figures & Tables

Builds every figure and table for the manuscript from the outputs of Phases 1–5.

**Deliverables** (all written to `results/publication/`):

| | |
|---|---|
| `main_figures/` | Figures 1–6 |
| `main_tables/` | Tables 1–4 |
| `supplementary_figures/` | Figures S1–S10 |
| `supplementary_tables/` | Tables S1–S10 |
| `CAPTIONS.md` | Draft caption for every panel |
| `FIGURE_INDEX.csv` | Machine-readable manifest |

Figures are vector **PDF** (600 dpi, TrueType fonts) with a **PNG** preview alongside.
Point clouds are rasterized inside the vector page — 121,869 individual vector points
would produce a PDF no editor can open.

## Setup

In [1]:
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import scanpy as sc
import anndata as ad
import scipy.sparse as sp

warnings.filterwarnings('ignore')
sc.settings.n_jobs = -1
sc.settings.verbosity = 1

import os
project_root = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
print(f"Project root: {project_root}")

Project root: /Users/jubayer/Projects/single-cell/fezf2-multiomics


### Publication style

One style block governs every figure so the set reads as a single coherent system.

In [2]:
# ---------------------------------------------------------------- page geometry
MM = 1 / 25.4
W_SINGLE = 89 * MM    # single column  (89 mm)
W_HALF   = 120 * MM   # 1.5 column
W_DOUBLE = 183 * MM   # full width     (183 mm)

# ---------------------------------------------------------------- rcParams
mpl.rcParams.update({
    'pdf.fonttype': 42,          # TrueType: journals require editable vector text
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 7,
    'axes.labelsize': 7,
    'axes.titlesize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 6,
    'legend.frameon': False,
    'axes.linewidth': 0.5,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 2,
    'ytick.major.size': 2,
    'lines.linewidth': 1.0,
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.transparent': False,
})

# ---------------------------------------------------------------- palettes
# Genotype: sequential dose (2 -> 1 -> 0 copies), colourblind-safe
GENOTYPE_COLORS = {'WT': '#3B75AF', 'Het': '#EF8636', 'KO': '#C53A32'}
GENOTYPE_ORDER = ['WT', 'Het', 'KO']

# Dose-response patterns
PATTERN_COLORS = {
    'Linear': '#3B75AF', 'Compensatory': '#C53A32', 'Threshold': '#519E3E',
    'Synergistic': '#8D69B8', 'No Response': '#BFBFBF',
}

# 14 cell types - qualitative, maximally separable
CELLTYPE_PALETTE = [
    '#3B75AF', '#EF8636', '#519E3E', '#C53A32', '#8D69B8',
    '#84584E', '#D57DBE', '#7F7F7F', '#BCBD22', '#17BECF',
    '#AEC7E8', '#FFBB78', '#98DF8A', '#FF9896',
]

SEQ_CMAP = 'viridis'
DIV_CMAP = 'RdBu_r'

TIMEPOINT_ORDER = ['E10', 'E11.5', 'E12.5', 'E13', 'E13.5', 'E14.5', 'E15',
                   'E15.5', 'E16', 'E17.5', 'E18.5', 'P1', 'P4']

# ---------------------------------------------------------------- output dirs
PUB = project_root / 'results' / 'publication'
FIG_MAIN = PUB / 'main_figures'
FIG_SUPP = PUB / 'supplementary_figures'
TAB_MAIN = PUB / 'main_tables'
TAB_SUPP = PUB / 'supplementary_tables'
for d in (FIG_MAIN, FIG_SUPP, TAB_MAIN, TAB_SUPP):
    d.mkdir(parents=True, exist_ok=True)

MANIFEST = []   # every artefact registered here -> FIGURE_INDEX.csv
CAPTIONS = {}   # name -> caption text

print(f"Publication output: {PUB}")

Publication output: /Users/jubayer/Projects/single-cell/fezf2-multiomics/results/publication


In [3]:
# ---------------------------------------------------------------- helpers

def panel_label(ax, letter, x=-0.16, y=1.04):
    """Bold panel letter in the top-left, in axes coordinates."""
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=8,
            fontweight='bold', va='bottom', ha='right')


def clean(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    return ax


def embed_ax(ax):
    """Strip an axis down for a UMAP/embedding panel."""
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xlabel(''); ax.set_ylabel('')
    return ax


def save_figure(fig, name, kind, caption):
    """Write PDF (vector, submission) + PNG (preview) and register in the manifest."""
    d = FIG_MAIN if kind == 'main' else FIG_SUPP
    fig.savefig(d / f'{name}.pdf')
    fig.savefig(d / f'{name}.png', dpi=300)
    plt.close(fig)
    CAPTIONS[name] = caption
    MANIFEST.append({'type': f'{kind} figure', 'name': name,
                     'file': f'{d.name}/{name}.pdf', 'caption': caption})
    print(f"  saved {name}")


def save_table(df, name, kind, caption, index=False):
    d = TAB_MAIN if kind == 'main' else TAB_SUPP
    df.to_csv(d / f'{name}.csv', index=index)
    CAPTIONS[name] = caption
    MANIFEST.append({'type': f'{kind} table', 'name': name,
                     'file': f'{d.name}/{name}.csv', 'caption': caption,
                     'n_rows': len(df)})
    print(f"  saved {name}  ({len(df):,} rows)")


def scatter_categorical(ax, xy, labels, palette, size=1.2, order=None, legend=False,
                        legend_kw=None, rasterized=True):
    """Embedding scatter coloured by a categorical variable.

    Points are rasterized: 121,869 vector points would make the PDF unopenable.
    """
    cats = order if order is not None else list(pd.unique(labels))
    for i, c in enumerate(cats):
        m = np.asarray(labels) == c
        if not m.any():
            continue
        col = palette[c] if isinstance(palette, dict) else palette[i % len(palette)]
        ax.scatter(xy[m, 0], xy[m, 1], s=size, c=col, linewidths=0,
                   rasterized=rasterized, label=str(c))
    embed_ax(ax)
    if legend:
        kw = dict(loc='center left', bbox_to_anchor=(1.0, 0.5), markerscale=4,
                  handletextpad=0.2, borderpad=0, labelspacing=0.35)
        kw.update(legend_kw or {})
        ax.legend(**kw)
    return ax


def scatter_continuous(ax, xy, values, cmap=SEQ_CMAP, size=1.2, label='',
                       vmin=None, vmax=None, cbar=True, order_by_value=True):
    """Embedding scatter coloured by a continuous variable (expressing cells on top)."""
    v = np.asarray(values, dtype=float)
    idx = np.argsort(v) if order_by_value else np.arange(len(v))
    s = ax.scatter(xy[idx, 0], xy[idx, 1], s=size, c=v[idx], cmap=cmap,
                   vmin=vmin, vmax=vmax, linewidths=0, rasterized=True)
    embed_ax(ax)
    if cbar:
        cb = plt.colorbar(s, ax=ax, fraction=0.035, pad=0.02)
        cb.set_label(label, fontsize=6)
        cb.ax.tick_params(labelsize=5, width=0.4, length=1.5)
        cb.outline.set_linewidth(0.4)
    return ax


def gene_vector(adata, gene, layer='log1p_norm'):
    """1-D dense expression vector for one gene (kept sparse until the last moment)."""
    X = adata.layers[layer] if layer in adata.layers else adata.X
    col = X[:, adata.var_names.get_loc(gene)]
    return np.asarray(col.todense()).ravel() if sp.issparse(col) else np.asarray(col).ravel()

print("helpers ready")

helpers ready


## Load Phase 1–5 outputs

In [4]:
adata = sc.read_h5ad(project_root / 'results' / 'temporal' / 'adata_annotated.h5ad')
print(f"scRNA-seq: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

# Stable, biologically ordered category orders -------------------------------
present_tp = [t for t in TIMEPOINT_ORDER if t in set(adata.obs['timepoint'])]
adata.obs['timepoint'] = pd.Categorical(adata.obs['timepoint'], categories=present_tp, ordered=True)
adata.obs['genotype'] = pd.Categorical(adata.obs['genotype'], categories=GENOTYPE_ORDER, ordered=True)

# Order cell types by developmental logic (progenitors -> neurons -> glia/other)
CELLTYPE_ORDER = [
    'Radial Glia', 'Cycling Progenitors', 'Intermediate Progenitors',
    'Subcerebral Projection Neurons', 'Corticothalamic Neurons',
    'Layer 4 Neurons', 'Layer 2 3 Neurons', 'Callosal Projection Neurons',
    'Cajal-Retzius Cells', 'GABAergic Interneurons',
    'Oligodendrocyte Precursors', 'Microglia', 'Endothelial', 'Pericytes',
]
observed = list(adata.obs['cell_type'].cat.categories) if hasattr(adata.obs['cell_type'], 'cat') \
    else list(pd.unique(adata.obs['cell_type']))
CELLTYPE_ORDER = [c for c in CELLTYPE_ORDER if c in observed] + \
                 [c for c in observed if c not in CELLTYPE_ORDER]
adata.obs['cell_type'] = pd.Categorical(adata.obs['cell_type'], categories=CELLTYPE_ORDER, ordered=True)

CT_COLORS = {ct: CELLTYPE_PALETTE[i % len(CELLTYPE_PALETTE)]
             for i, ct in enumerate(CELLTYPE_ORDER)}

UMAP = adata.obsm['X_umap']
print(f"cell types: {len(CELLTYPE_ORDER)} | timepoints: {len(present_tp)}")

scRNA-seq: 121,869 cells x 3,000 genes
cell types: 14 | timepoints: 13


In [5]:
def read_if(path, **kw):
    p = project_root / path
    if p.exists():
        return pd.read_csv(p, **kw)
    print(f"  MISSING: {path}")
    return None

dose_df      = read_if('results/dose_response/gene_classifications/dose_response_p1.csv')
sex_de       = read_if('results/dose_response/sex_dimorphism/sex_de_het_p1.csv')
de_wt_ko     = read_if('results/temporal/de_results/de_wt_vs_ko_p1.csv')
cluster_mark = read_if('results/temporal/annotations/cluster_markers.csv')
fezf2_tg     = read_if('results/multiomics/networks/fezf2_targets_e13.csv')
drug_int     = read_if('results/therapeutic/drug_targets/drug_gene_interactions.csv')
targets      = read_if('results/therapeutic/drug_targets/prioritized_targets.csv')
integ_metrics= read_if('results/preprocessing/qc_reports/integration_comparison_metrics.csv')
atac         = read_if('results/multiomics/atac_embedding.csv')

if dose_df is not None:
    dose_df['pattern'] = pd.Categorical(
        dose_df['pattern'],
        categories=[p for p in PATTERN_COLORS if p in set(dose_df['pattern'])], ordered=True)

# All TF networks from Phase 4
tf_dir = project_root / 'results/multiomics/networks'
tf_networks = {}
for f in sorted(tf_dir.glob('*_targets_e13.csv')):
    tf = f.stem.replace('_targets_e13', '')
    d = pd.read_csv(f)
    tf_networks[tf] = d
print(f"TF networks: {len(tf_networks)} | ATAC cells: {0 if atac is None else len(atac):,}")

TF networks: 15 | ATAC cells: 31,663


---
## Figure 1 — Single-cell atlas of Fezf2-dependent corticogenesis

In [6]:
fig = plt.figure(figsize=(W_DOUBLE, 0.92 * W_DOUBLE))
gs = fig.add_gridspec(3, 3, height_ratios=[1.15, 1.0, 0.9], hspace=0.85, wspace=0.30)

# Legends sit BELOW each embedding: an inline legend is painted over by the next
# axes' background, and a right-hand legend collides with the neighbouring panel.
BELOW = dict(loc='upper center', bbox_to_anchor=(0.5, -0.01), markerscale=4,
             handletextpad=0.15, columnspacing=0.5, labelspacing=0.3, borderpad=0)

# A - cell type
axA = fig.add_subplot(gs[0, 0])
scatter_categorical(axA, UMAP, adata.obs['cell_type'], CT_COLORS, order=CELLTYPE_ORDER,
                    legend=True, legend_kw=dict(ncol=2, fontsize=4.5, **BELOW))
axA.set_title('Cell type', pad=2); panel_label(axA, 'A', x=-0.02)

# B - timepoint
axB = fig.add_subplot(gs[0, 1])
tp_cmap = plt.get_cmap('viridis', len(present_tp))
tp_colors = {tp: tp_cmap(i) for i, tp in enumerate(present_tp)}
scatter_categorical(axB, UMAP, adata.obs['timepoint'], tp_colors, order=present_tp,
                    legend=True, legend_kw=dict(ncol=4, fontsize=4.5, **BELOW))
axB.set_title('Developmental stage', pad=2); panel_label(axB, 'B', x=-0.02)

# C - genotype
axC = fig.add_subplot(gs[0, 2])
scatter_categorical(axC, UMAP, adata.obs['genotype'], GENOTYPE_COLORS, order=GENOTYPE_ORDER,
                    legend=True, legend_kw=dict(ncol=3, fontsize=5.5, **BELOW))
axC.set_title('Genotype', pad=2); panel_label(axC, 'C', x=-0.02)

# D - marker dot plot (mean z-scored expression + % expressing)
MARKERS = {
    'Radial Glia': ['Sox2', 'Pax6', 'Fabp7', 'Slc1a3'],
    'Cycling Progenitors': ['Mki67', 'Top2a', 'Ccnb1'],
    'Intermediate Progenitors': ['Eomes', 'Neurog2', 'Btg2'],
    'Subcerebral Projection Neurons': ['Fezf2', 'Bcl11b', 'Crym'],
    'Corticothalamic Neurons': ['Tle4', 'Syt6', 'Nr4a2'],
    'Layer 4 Neurons': ['Rorb', 'Whrn'],
    'Layer 2 3 Neurons': ['Cux2', 'Satb2', 'Mef2c'],
    'Cajal-Retzius Cells': ['Reln', 'Trp73', 'Lhx5'],
    'GABAergic Interneurons': ['Gad1', 'Gad2', 'Dlx2'],
    'Oligodendrocyte Precursors': ['Pdgfra', 'Olig1'],
    'Microglia': ['Aif1', 'C1qa', 'Cx3cr1'],
    'Endothelial': ['Cldn5', 'Pecam1'],
    'Pericytes': ['Pdgfrb', 'Vtn'],
}
marker_genes, seen = [], set()
for genes in MARKERS.values():
    for g in genes:
        if g in adata.var_names and g not in seen:
            marker_genes.append(g); seen.add(g)

expr = pd.DataFrame({g: gene_vector(adata, g) for g in marker_genes})
expr['cell_type'] = adata.obs['cell_type'].values
mean_expr = expr.groupby('cell_type', observed=True)[marker_genes].mean()
pct_expr = expr.groupby('cell_type', observed=True)[marker_genes].apply(lambda d: (d > 0).mean() * 100)
z = ((mean_expr - mean_expr.mean()) / mean_expr.std().replace(0, 1)).reindex(CELLTYPE_ORDER)
pct_expr = pct_expr.reindex(CELLTYPE_ORDER)

axD = fig.add_subplot(gs[1, :])
xs, ys, cs, ss = [], [], [], []
for yi, ct in enumerate(z.index):
    for xi, g in enumerate(marker_genes):
        xs.append(xi); ys.append(yi); cs.append(z.loc[ct, g]); ss.append(pct_expr.loc[ct, g])
sc_d = axD.scatter(xs, ys, c=cs, s=np.array(ss) * 0.30, cmap=DIV_CMAP,
                   vmin=-2, vmax=2, linewidths=0.2, edgecolors='#333333')
axD.set_xticks(range(len(marker_genes)))
axD.set_xticklabels(marker_genes, rotation=90, style='italic', fontsize=5)
axD.set_yticks(range(len(z.index))); axD.set_yticklabels(z.index, fontsize=5.5)
axD.invert_yaxis(); axD.set_xlim(-0.8, len(marker_genes) - 0.2)
axD.tick_params(length=0)
for s in axD.spines.values():
    s.set_visible(False)
axD.grid(True, lw=0.25, color='#E8E8E8', zorder=0); axD.set_axisbelow(True)
cb = plt.colorbar(sc_d, ax=axD, fraction=0.011, pad=0.012)
cb.set_label('Mean expression (z-score)', fontsize=5.5, labelpad=1)
cb.ax.tick_params(labelsize=5, width=0.4, length=1.5); cb.outline.set_linewidth(0.4)
handles = [Line2D([], [], marker='o', ls='', markersize=np.sqrt(p * 0.30),
                  markerfacecolor='#888888', markeredgecolor='#333333', markeredgewidth=0.2,
                  label=f'{p}%') for p in (25, 50, 75, 100)]
# pushed clear of the colourbar, which otherwise sits underneath it
axD.legend(handles=handles, title='% cells', loc='upper left', bbox_to_anchor=(1.075, 1.0),
           fontsize=5, title_fontsize=5.5, labelspacing=0.7, handletextpad=0.4)
axD.set_title('Canonical marker gene expression by annotated cell type', pad=4)
panel_label(axD, 'D', x=-0.075)

# E - composition across development
axE = fig.add_subplot(gs[2, :2])
comp = pd.crosstab(adata.obs['timepoint'], adata.obs['cell_type'], normalize='index') * 100
comp = comp.reindex(index=present_tp, columns=CELLTYPE_ORDER)
bottom = np.zeros(len(comp))
for ct in CELLTYPE_ORDER:
    axE.bar(range(len(comp)), comp[ct].values, bottom=bottom, width=0.82,
            color=CT_COLORS[ct], linewidth=0, label=ct)
    bottom += comp[ct].values
axE.set_xticks(range(len(comp))); axE.set_xticklabels(comp.index, rotation=45, ha='right')
axE.set_ylabel('Composition (%)'); axE.set_xlabel('Developmental stage')
axE.set_ylim(0, 100); axE.set_xlim(-0.6, len(comp) - 0.4); clean(axE)
axE.set_title('Cell type composition across corticogenesis (colours as in A)', pad=3)
panel_label(axE, 'E', x=-0.075)

# F - cells per sample
axF = fig.add_subplot(gs[2, 2])
n_by_geno = adata.obs.groupby(['timepoint', 'genotype'], observed=True).size().unstack(fill_value=0)
n_by_geno = n_by_geno.reindex(index=present_tp, columns=GENOTYPE_ORDER, fill_value=0)
x = np.arange(len(n_by_geno)); w = 0.27
for i, g in enumerate(GENOTYPE_ORDER):
    axF.bar(x + (i - 1) * w, n_by_geno[g].values / 1000, width=w,
            color=GENOTYPE_COLORS[g], linewidth=0, label=g)
axF.set_xticks(x); axF.set_xticklabels(n_by_geno.index, rotation=45, ha='right', fontsize=5)
axF.set_ylabel('Cells (x10$^3$)'); axF.set_xlabel('Developmental stage')
axF.legend(title='Genotype', title_fontsize=6, loc='upper left'); clean(axF)
axF.set_title('Sampling depth by genotype', pad=3)
panel_label(axF, 'F', x=-0.18)

save_figure(fig, 'Figure1_single_cell_atlas', 'main',
    "Figure 1. Single-cell atlas of Fezf2-dependent corticogenesis. "
    f"(A) UMAP of {adata.n_obs:,} cells coloured by the {len(CELLTYPE_ORDER)} annotated cell types. "
    "(B) The same embedding coloured by developmental stage (E10-P4). "
    "(C) The same embedding coloured by Fezf2 genotype (WT, Het, KO). "
    "(D) Dot plot of canonical marker genes supporting each annotation; dot colour is mean "
    "expression z-scored across cell types and dot size is the percentage of cells expressing. "
    "(E) Stacked composition of cell types across developmental stages. "
    "(F) Number of cells recovered per stage and genotype; note that WT animals were not "
    "sampled at E13 or E15.")

  saved Figure1_single_cell_atlas


---
## Figure 2 — Fezf2 expression and knockout validation

In [7]:
fezf2 = gene_vector(adata, 'Fezf2')
adata.obs['Fezf2_expr'] = fezf2

fig = plt.figure(figsize=(W_DOUBLE, 0.52 * W_DOUBLE))
gs = fig.add_gridspec(2, 3, hspace=0.55, wspace=0.4)

# A - Fezf2 on the UMAP
axA = fig.add_subplot(gs[0, 0])
scatter_continuous(axA, UMAP, fezf2, cmap='Reds', label='Fezf2 (log-norm)')
axA.set_title('$\\it{Fezf2}$ expression', pad=2); panel_label(axA, 'A', x=-0.02)

# B - by cell type
axB = fig.add_subplot(gs[0, 1:])
data = [fezf2[(adata.obs['cell_type'] == ct).values] for ct in CELLTYPE_ORDER]
parts = axB.violinplot(data, showextrema=False, widths=0.85)
for pc, ct in zip(parts['bodies'], CELLTYPE_ORDER):
    pc.set_facecolor(CT_COLORS[ct]); pc.set_alpha(0.9); pc.set_linewidth(0)
means = [d.mean() for d in data]
axB.scatter(range(1, len(data) + 1), means, s=4, color='#222222', zorder=3, linewidths=0)
axB.set_xticks(range(1, len(CELLTYPE_ORDER) + 1))
axB.set_xticklabels(CELLTYPE_ORDER, rotation=40, ha='right', fontsize=5.5)
axB.set_ylabel('$\\it{Fezf2}$ (log-norm)'); clean(axB)
axB.set_title('$\\it{Fezf2}$ is restricted to subcerebral projection neurons', pad=3)
panel_label(axB, 'B', x=-0.06)

# C - Fezf2 over developmental time (WT only)
axC = fig.add_subplot(gs[1, 0])
wt = adata.obs['genotype'] == 'WT'
tp_mean = pd.DataFrame({'tp': adata.obs['timepoint'][wt].values, 'e': fezf2[wt.values]}) \
    .groupby('tp', observed=True)['e'].agg(['mean', 'sem']).reindex(present_tp).dropna()
axC.errorbar(range(len(tp_mean)), tp_mean['mean'], yerr=tp_mean['sem'],
             marker='o', ms=3, color=GENOTYPE_COLORS['WT'], lw=1, capsize=1.5, elinewidth=0.5)
axC.set_xticks(range(len(tp_mean))); axC.set_xticklabels(tp_mean.index, rotation=45, ha='right', fontsize=5)
axC.set_ylabel('Mean $\\it{Fezf2}$'); axC.set_xlabel('Developmental stage'); clean(axC)
axC.set_title('$\\it{Fezf2}$ dynamics (WT)', pad=3); panel_label(axC, 'C', x=-0.22)

# D - dosage validation at P1 (the only 3-genotype stage)
axD = fig.add_subplot(gs[1, 1])
p1 = adata.obs['timepoint'] == 'P1'
vals, labs = [], []
for g in GENOTYPE_ORDER:
    m = (p1 & (adata.obs['genotype'] == g)).values
    if m.sum():
        vals.append(fezf2[m]); labs.append(g)
bp = axD.boxplot(vals, widths=0.55, patch_artist=True, showfliers=False,
                 medianprops=dict(color='black', lw=0.8),
                 boxprops=dict(lw=0.5), whiskerprops=dict(lw=0.5), capprops=dict(lw=0.5))
for patch, g in zip(bp['boxes'], labs):
    patch.set_facecolor(GENOTYPE_COLORS[g]); patch.set_alpha(0.85)
axD.set_xticklabels(labs); axD.set_ylabel('$\\it{Fezf2}$ (log-norm)')
axD.set_xlabel('Genotype'); clean(axD)
axD.set_title('$\\it{Fezf2}$ dosage at P1', pad=3); panel_label(axD, 'D', x=-0.22)

# E - fraction of Fezf2+ cells by genotype at P1
axE = fig.add_subplot(gs[1, 2])
frac = [100 * (v > 0).mean() for v in vals]
axE.bar(range(len(labs)), frac, width=0.6,
        color=[GENOTYPE_COLORS[g] for g in labs], linewidth=0)
for i, f in enumerate(frac):
    axE.text(i, f + 0.6, f'{f:.1f}%', ha='center', fontsize=5.5)
axE.set_xticks(range(len(labs))); axE.set_xticklabels(labs)
axE.set_ylabel('$\\it{Fezf2}$-positive cells (%)'); axE.set_xlabel('Genotype')
axE.set_ylim(0, max(frac) * 1.25); clean(axE)
axE.set_title('$\\it{Fezf2}$-positive fraction at P1', pad=3); panel_label(axE, 'E', x=-0.22)

save_figure(fig, 'Figure2_fezf2_expression', 'main',
    "Figure 2. Fezf2 expression, cell-type specificity and knockout validation. "
    "(A) Fezf2 expression projected onto the UMAP; expressing cells are plotted last. "
    "(B) Fezf2 expression by cell type (violins; black dots, means), confirming enrichment "
    "in subcerebral projection neurons. "
    "(C) Mean Fezf2 expression across developmental stages in WT animals (error bars, s.e.m.). "
    "(D) Fezf2 expression by genotype at P1, the only stage sampling all three genotypes. "
    "(E) Percentage of Fezf2-positive cells per genotype at P1.")
del expr; gc.collect()

  saved Figure2_fezf2_expression


4859

---
## Figure 3 — Developmental trajectories and pseudotime

In [8]:
fig = plt.figure(figsize=(W_DOUBLE, 0.50 * W_DOUBLE))
gs = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.4)

# A - PAGA lineage graph
axA = fig.add_subplot(gs[0, 0])
try:
    sc.pl.paga(adata, threshold=0.15, node_size_scale=1.6, fontsize=4,
               ax=axA, show=False, frameon=False, edge_width_scale=0.4,
               colors='cell_type')
    axA.set_title('PAGA lineage graph', pad=2)
except Exception as e:
    print(f"PAGA panel unavailable: {e}")
    scatter_categorical(axA, UMAP, adata.obs['cell_type'], CT_COLORS, order=CELLTYPE_ORDER)
    axA.set_title('Cell type', pad=2)
embed_ax(axA); panel_label(axA, 'A', x=-0.02)

# B - pseudotime on the UMAP
axB = fig.add_subplot(gs[0, 1])
has_pt = 'dpt_pseudotime' in adata.obs
if has_pt:
    scatter_continuous(axB, UMAP, adata.obs['dpt_pseudotime'].values, cmap='magma',
                       label='Pseudotime')
axB.set_title('Diffusion pseudotime', pad=2); panel_label(axB, 'B', x=-0.02)

# C - pseudotime vs real developmental time
axC = fig.add_subplot(gs[0, 2])
if has_pt:
    pt = pd.DataFrame({'tp': adata.obs['timepoint'].values,
                       'pt': adata.obs['dpt_pseudotime'].values}).dropna()
    data = [pt.loc[pt['tp'] == t, 'pt'].values for t in present_tp]
    bp = axC.boxplot(data, widths=0.6, patch_artist=True, showfliers=False,
                     medianprops=dict(color='black', lw=0.6), boxprops=dict(lw=0.4),
                     whiskerprops=dict(lw=0.4), capprops=dict(lw=0.4))
    for patch, i in zip(bp['boxes'], range(len(present_tp))):
        patch.set_facecolor(tp_cmap(i)); patch.set_alpha(0.9)
    axC.set_xticklabels(present_tp, rotation=45, ha='right', fontsize=5)
    axC.set_ylabel('Pseudotime'); axC.set_xlabel('Developmental stage')
    r = np.corrcoef(pd.Categorical(pt['tp'], categories=present_tp, ordered=True).codes, pt['pt'])[0, 1]
    axC.text(0.03, 0.95, f'r = {r:.2f}', transform=axC.transAxes, fontsize=6, va='top')
clean(axC); axC.set_title('Pseudotime recovers real time', pad=3); panel_label(axC, 'C', x=-0.2)

# D - pseudotime distribution by genotype
axD = fig.add_subplot(gs[1, 0])
if has_pt:
    for g in GENOTYPE_ORDER:
        v = adata.obs.loc[adata.obs['genotype'] == g, 'dpt_pseudotime'].dropna()
        if len(v):
            axD.hist(v, bins=40, density=True, histtype='step', lw=1,
                     color=GENOTYPE_COLORS[g], label=f'{g} (n={len(v):,})')
    axD.legend(loc='upper right')
axD.set_xlabel('Pseudotime'); axD.set_ylabel('Density'); clean(axD)
axD.set_title('Trajectory divergence by genotype', pad=3); panel_label(axD, 'D', x=-0.2)

# E - cell type occupancy along pseudotime
axE = fig.add_subplot(gs[1, 1:])
if has_pt:
    ptv = adata.obs['dpt_pseudotime'].values
    ok = np.isfinite(ptv)
    bins = np.linspace(np.nanmin(ptv), np.nanmax(ptv), 26)
    binned = pd.cut(ptv[ok], bins, include_lowest=True)
    occ = pd.crosstab(binned, adata.obs['cell_type'].values[ok], normalize='index') * 100
    occ = occ.reindex(columns=CELLTYPE_ORDER, fill_value=0)
    centers = (bins[:-1] + bins[1:]) / 2
    axE.stackplot(centers, [occ[ct].values for ct in CELLTYPE_ORDER],
                  colors=[CT_COLORS[ct] for ct in CELLTYPE_ORDER], linewidth=0)
    axE.set_xlim(centers[0], centers[-1]); axE.set_ylim(0, 100)
    axE.set_xlabel('Pseudotime'); axE.set_ylabel('Composition (%)')
    axE.legend([Patch(facecolor=CT_COLORS[ct]) for ct in CELLTYPE_ORDER], CELLTYPE_ORDER,
               loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=5, ncol=1)
clean(axE); axE.set_title('Cell identity along the differentiation axis', pad=3)
panel_label(axE, 'E', x=-0.06)

save_figure(fig, 'Figure3_trajectories', 'main',
    "Figure 3. Developmental trajectories and pseudotemporal ordering. "
    "(A) PAGA graph abstracting lineage relationships between cell types. "
    "(B) Diffusion pseudotime projected onto the UMAP. "
    "(C) Pseudotime against true developmental stage (boxes, median and IQR); the Pearson "
    "correlation is annotated. "
    "(D) Pseudotime distribution per genotype. "
    "(E) Cell type composition in 25 equal-width pseudotime bins, tracing progenitors "
    "through to differentiated neurons.")

  saved Figure3_trajectories


---
## Figure 4 — Dose-dependent transcriptional response to Fezf2 loss

In [9]:
fig = plt.figure(figsize=(W_DOUBLE, 0.56 * W_DOUBLE))
gs = fig.add_gridspec(2, 3, hspace=0.55, wspace=0.42)
pat_order = [p for p in PATTERN_COLORS if p in set(dose_df['pattern'])]

# A - pattern counts
axA = fig.add_subplot(gs[0, 0])
counts = dose_df['pattern'].value_counts().reindex(pat_order)
axA.bar(range(len(counts)), counts.values, width=0.68,
        color=[PATTERN_COLORS[p] for p in counts.index], linewidth=0)
for i, v in enumerate(counts.values):
    axA.text(i, v + 25, f'{v:,}', ha='center', fontsize=5.5)
axA.set_xticks(range(len(counts)))
axA.set_xticklabels(counts.index, rotation=40, ha='right', fontsize=5.5)
axA.set_ylabel('Genes'); axA.set_ylim(0, counts.max() * 1.16); clean(axA)
axA.set_title('Dose-response classification (P1)', pad=3); panel_label(axA, 'A', x=-0.2)

# B - Het vs KO fold-change space
axB = fig.add_subplot(gs[0, 1])
for p in pat_order:
    d = dose_df[dose_df['pattern'] == p]
    axB.scatter(d['Het_vs_WT_fc'], d['KO_vs_WT_fc'], s=2.5, alpha=0.65,
                c=PATTERN_COLORS[p], linewidths=0, rasterized=True, label=p)
lim = np.nanpercentile(np.abs(dose_df[['Het_vs_WT_fc', 'KO_vs_WT_fc']].values), 99.5)
axB.plot([-lim, lim], [-lim, lim], ls='--', lw=0.5, color='#555555', zorder=0)
axB.axhline(0, lw=0.4, color='#999999', zorder=0); axB.axvline(0, lw=0.4, color='#999999', zorder=0)
axB.set_xlim(-lim, lim); axB.set_ylim(-lim, lim)
axB.set_xlabel('Het vs WT (log$_2$ FC)'); axB.set_ylabel('KO vs WT (log$_2$ FC)')
axB.legend(markerscale=2.5, loc='upper left', fontsize=5); clean(axB)
axB.set_title('Dose-dependent effect space', pad=3); panel_label(axB, 'B', x=-0.2)

# C - representative dose curves
axC = fig.add_subplot(gs[0, 2])
for p in ['Linear', 'Compensatory', 'Threshold']:
    d = dose_df[dose_df['pattern'] == p]
    if not len(d):
        continue
    top = d.loc[d['KO_vs_WT_fc'].abs().nlargest(8).index]
    curves = top[['WT_mean', 'Het_mean', 'KO_mean']].values
    curves = curves - curves[:, [0]]           # centre each gene on its WT level
    axC.plot([0, 1, 2], curves.T, color=PATTERN_COLORS[p], alpha=0.32, lw=0.7)
    axC.plot([0, 1, 2], curves.mean(axis=0), color=PATTERN_COLORS[p], lw=1.8, label=p)
axC.set_xticks([0, 1, 2]); axC.set_xticklabels(['WT\n(2 copies)', 'Het\n(1)', 'KO\n(0)'], fontsize=5.5)
axC.axhline(0, lw=0.4, color='#999999', zorder=0)
axC.set_ylabel('$\\Delta$ expression vs WT'); axC.set_xlabel('$\\it{Fezf2}$ dosage')
axC.legend(loc='best', fontsize=5.5); clean(axC)
axC.set_title('Representative dose curves', pad=3); panel_label(axC, 'C', x=-0.22)

# D - volcano, WT vs KO at P1
axD = fig.add_subplot(gs[1, 0])
if de_wt_ko is not None:
    v = de_wt_ko.dropna(subset=['logfoldchanges', 'pvals_adj']).copy()
    # scanpy's t-test emits extreme log-fold-changes for near-zero-expression genes
    # (up to |80|). Left unclipped the axis is unreadable; clip the display only.
    LFC_CLIP, NLP_CLIP = 8.0, 300.0
    v['x'] = v['logfoldchanges'].clip(-LFC_CLIP, LFC_CLIP)
    v['nlp'] = (-np.log10(v['pvals_adj'].clip(lower=1e-300))).clip(upper=NLP_CLIP)
    sig = (v['pvals_adj'] < 0.05) & (v['logfoldchanges'].abs() > 1)
    axD.scatter(v.loc[~sig, 'x'], v.loc[~sig, 'nlp'], s=1.5,
                c='#CCCCCC', linewidths=0, rasterized=True)
    axD.scatter(v.loc[sig, 'x'], v.loc[sig, 'nlp'], s=1.8,
                c='#C53A32', linewidths=0, rasterized=True)
    axD.axhline(-np.log10(0.05), ls='--', lw=0.4, color='#555555')
    for xv in (-1, 1):
        axD.axvline(xv, ls='--', lw=0.4, color='#555555')
    # Label by |effect size|, not by p-value: the top p-values all pile up at the same
    # y and their labels overprint each other.
    lab = v.loc[sig].reindex(v.loc[sig, 'logfoldchanges'].abs().nlargest(7).index)
    for _, r in lab.iterrows():
        axD.annotate(r['names'], (r['x'], r['nlp']), fontsize=4.5, style='italic',
                     xytext=(0, 4), textcoords='offset points', ha='center')
    axD.set_xlim(-LFC_CLIP - 0.5, LFC_CLIP + 0.5)
    axD.set_ylim(-8, NLP_CLIP * 1.16)
    axD.set_xlabel('log$_2$ FC (KO vs WT), clipped to $\\pm$8')
    axD.set_ylabel('-log$_{10}$ FDR (clipped)')
    axD.text(0.03, 0.95, f'{int(sig.sum()):,} DE genes', transform=axD.transAxes,
             ha='left', va='top', fontsize=5.5)
clean(axD); axD.set_title('Differential expression, P1', pad=3); panel_label(axD, 'D', x=-0.2)

# E - compensatory magnitude
axE = fig.add_subplot(gs[1, 1])
comp = dose_df[dose_df['pattern'] == 'Compensatory'].nlargest(18, 'KO_vs_WT_fc')
axE.barh(range(len(comp)), comp['KO_vs_WT_fc'].values, height=0.72,
         color=PATTERN_COLORS['Compensatory'], linewidth=0)
axE.set_yticks(range(len(comp)))
axE.set_yticklabels(comp['gene'], fontsize=5, style='italic')
axE.invert_yaxis(); axE.set_xlabel('log$_2$ FC (KO vs WT)'); clean(axE)
axE.set_title('Top compensatory genes', pad=3); panel_label(axE, 'E', x=-0.32)

# F - dosage correlation is QUANTISED: Spearman over 3 genotype means can only take
# the values {-1, -0.5, 0, 0.5, 1}. A histogram/KDE of it is misleading (it renders as
# spikes and implies a continuum), so show the discrete support honestly.
axF = fig.add_subplot(gs[1, 2])
levels = [-1.0, -0.5, 0.0, 0.5, 1.0]
rho = dose_df['dose_correlation'].round(1)
ct_rho = pd.crosstab(rho, dose_df['pattern']).reindex(index=levels, columns=pat_order, fill_value=0)
x = np.arange(len(levels)); w = 0.8 / max(len(pat_order), 1)
for i, p in enumerate(pat_order):
    axF.bar(x + (i - (len(pat_order) - 1) / 2) * w, ct_rho[p].values, width=w,
            color=PATTERN_COLORS[p], linewidth=0, label=p)
axF.set_xticks(x); axF.set_xticklabels([f'{l:+.1f}'.replace('+0.0', '0.0') for l in levels])
axF.set_xlabel('Spearman r with $\\it{Fezf2}$ dosage')
axF.set_ylabel('Genes')
axF.legend(fontsize=5, loc='upper center', ncol=2); clean(axF)
axF.set_title('Dosage correlation (quantised, n = 3)', pad=3); panel_label(axF, 'F', x=-0.22)

save_figure(fig, 'Figure4_dose_response', 'main',
    "Figure 4. Dose-dependent transcriptional response to progressive Fezf2 loss at P1. "
    "(A) Genes classified by dose-response pattern. "
    "(B) Het-versus-KO log2 fold-change space, coloured by class; the dashed diagonal marks a "
    "purely linear response. "
    "(C) Expression trajectories of the 8 strongest genes per class across the WT-Het-KO dosage "
    "series, centred on the WT level (bold line, class mean). "
    "(D) Volcano plot of KO versus WT differential expression at P1 (red, FDR < 0.05 and "
    "|log2 FC| > 1). "
    "(E) The 18 most strongly up-regulated compensatory genes. "
    "(F) Spearman correlation with Fezf2 dosage. Because the correlation is computed over only "
    "three genotype means, it can take just five values; compensatory genes are negatively "
    "correlated, i.e. they rise as Fezf2 is lost. "
    "Note: P1 is the only stage at which all three genotypes were sampled, and the dose "
    "statistics rest on three pseudobulk means, so these classes are descriptive rather than "
    "inferential.")

  saved Figure4_dose_response


---
## Figure 5 — Chromatin accessibility and gene regulatory networks

In [10]:
import networkx as nx

fig = plt.figure(figsize=(W_DOUBLE, 0.50 * W_DOUBLE))
gs = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.4)

# A/B - scATAC embedding
if atac is not None:
    axy = atac[['UMAP1', 'UMAP2']].values
    axA = fig.add_subplot(gs[0, 0])
    atac_tps = sorted(atac['timepoint'].unique())
    at_cmap = plt.get_cmap('viridis', len(atac_tps))
    scatter_categorical(axA, axy, atac['timepoint'].values,
                        {t: at_cmap(i) for i, t in enumerate(atac_tps)},
                        order=atac_tps, size=1.0, legend=True, legend_kw=dict(fontsize=5.5))
    axA.set_title(f'scATAC-seq ({len(atac):,} cells)', pad=2); panel_label(axA, 'A', x=-0.02)

    axB = fig.add_subplot(gs[0, 1])
    clusters = sorted(atac['leiden'].astype(str).unique(), key=lambda s: int(s))
    cl_cmap = plt.get_cmap('tab20', len(clusters))
    scatter_categorical(axB, axy, atac['leiden'].astype(str).values,
                        {c: cl_cmap(i) for i, c in enumerate(clusters)},
                        order=clusters, size=1.0)
    axB.set_title(f'Accessibility clusters (n={len(clusters)})', pad=2)
    panel_label(axB, 'B', x=-0.02)

    axC = fig.add_subplot(gs[0, 2])
    for i, t in enumerate(atac_tps):
        axC.hist(np.log10(atac.loc[atac['timepoint'] == t, 'n_peaks'] + 1), bins=40,
                 histtype='step', lw=1, color=at_cmap(i), label=t, density=True)
    axC.set_xlabel('log$_{10}$ accessible peaks per cell'); axC.set_ylabel('Density')
    axC.legend(fontsize=5.5); clean(axC)
    axC.set_title('ATAC library complexity', pad=3); panel_label(axC, 'C', x=-0.2)

# D - Fezf2 target network
axD = fig.add_subplot(gs[1, 0])
if fezf2_tg is not None and len(fezf2_tg):
    G = nx.DiGraph()
    G.add_node('Fezf2')
    top = fezf2_tg.loc[fezf2_tg['correlation'].abs().nlargest(24).index]
    for _, r in top.iterrows():
        G.add_edge('Fezf2', r['gene'], w=abs(r['correlation']))
    pos = nx.spring_layout(G, k=0.9, seed=42)
    nx.draw_networkx_edges(G, pos, ax=axD, width=[G[u][v]['w'] * 3 for u, v in G.edges()],
                           edge_color='#BBBBBB', arrows=False)
    nx.draw_networkx_nodes(G, pos, ax=axD, nodelist=[n for n in G if n != 'Fezf2'],
                           node_color='#AEC7E8', node_size=90, linewidths=0.4,
                           edgecolors='#33638D')
    nx.draw_networkx_nodes(G, pos, ax=axD, nodelist=['Fezf2'], node_color='#C53A32',
                           node_size=260, linewidths=0.5, edgecolors='black')
    nx.draw_networkx_labels(G, pos, ax=axD, font_size=4.5, font_family='sans-serif')
    axD.text(0.5, -0.06, f"{len(fezf2_tg)} target(s) at |r| > 0.3", transform=axD.transAxes,
             ha='center', fontsize=5.5, color='#555555')
axD.axis('off')
axD.set_title('$\\it{Fezf2}$ correlation network (E13.5 WT)', pad=6, fontsize=6.5)
panel_label(axD, 'D', x=0.0, y=1.02)

# E - network size across TFs.
# The per-TF CSVs are capped at 100 rows by Phase 4's max_targets, so len(csv) is a
# censored count. Use the uncapped totals written to tf_network_summary.csv.
axE = fig.add_subplot(gs[1, 1:])
summ_path = project_root / 'results/multiomics/networks/tf_network_summary.csv'
if summ_path.exists():
    summ = pd.read_csv(summ_path).sort_values('n_targets_total', ascending=False)
    names = [t[:1].upper() + t[1:] for t in summ['TF']]   # the Fezf2 file is lower-cased on disk
    vals = summ['n_targets_total'].values
    capped = summ['n_targets_saved'].values
    ylab = 'Correlated targets (|r| > 0.3)'
else:
    s_ = pd.Series({tf: len(d) for tf, d in tf_networks.items()}).sort_values(ascending=False)
    names = [t[:1].upper() + t[1:] for t in s_.index]
    vals = s_.values; capped = vals
    ylab = 'Correlated targets (capped at 100)'

cols = ['#C53A32' if n.lower() == 'fezf2' else '#3B75AF' for n in names]
axE.bar(range(len(vals)), vals, width=0.7, color=cols, linewidth=0)
for i, (v_, c_) in enumerate(zip(vals, capped)):
    if v_ > c_:   # mark bars whose CSV was truncated
        axE.text(i, v_ * 1.02, '*', ha='center', fontsize=6, color='#555555')
axE.set_xticks(range(len(names)))
axE.set_xticklabels(names, rotation=45, ha='right', fontsize=5.5, style='italic')
axE.set_ylabel(ylab); axE.set_xlabel('Transcription factor')
axE.text(0.99, 0.95, '* only the top 100 targets are exported', transform=axE.transAxes,
         ha='right', va='top', fontsize=5, color='#555555')
clean(axE)
axE.set_title('Regulatory network size across cortical transcription factors', pad=3)
panel_label(axE, 'E', x=-0.06)

save_figure(fig, 'Figure5_chromatin_and_grn', 'main',
    "Figure 5. Chromatin accessibility and transcription-factor regulatory networks. "
    f"(A) UMAP of {0 if atac is None else len(atac):,} scATAC-seq cells (E13.5, E15.5, E18.5; all WT) "
    "coloured by stage. Cells separate almost completely by stage: the three ATAC libraries were "
    "not batch-integrated, so stage and batch are confounded and the accessibility clusters in (B) "
    "should not be read as cell types. "
    "(B) The same embedding coloured by accessibility cluster. "
    "(C) Distribution of accessible peaks recovered per cell, per stage. "
    "(D) Fezf2 correlation network at E13.5 in WT; edge width scales with |r|. Only two genes pass "
    "the |r| > 0.3 threshold, so this network is extremely sparse. "
    "(E) Number of correlated targets per transcription factor (Fezf2 in red); asterisks mark TFs "
    "whose exported target table was truncated at 100 genes. Networks are correlation-based and "
    "therefore associative, not causal.")

  saved Figure5_chromatin_and_grn


---
## Figure 6 — Therapeutic target prioritisation

In [11]:
fig = plt.figure(figsize=(W_DOUBLE, 0.50 * W_DOUBLE))
gs = fig.add_gridspec(2, 3, hspace=0.55, wspace=0.42)

drugged = targets[targets['has_drugs'] == 1].copy() if targets is not None else pd.DataFrame()

# A - top targets
axA = fig.add_subplot(gs[0, 0])
top = targets.nlargest(15, 'total_score')
axA.barh(range(len(top)), top['total_score'], height=0.72, color='#C53A32', linewidth=0)
axA.set_yticks(range(len(top))); axA.set_yticklabels(top['gene'], fontsize=5.5, style='italic')
axA.invert_yaxis(); axA.set_xlabel('Priority score'); clean(axA)
axA.set_title('Top prioritised targets', pad=3); panel_label(axA, 'A', x=-0.34)

# B - druggability vs effect size
axB = fig.add_subplot(gs[0, 1])
s = axB.scatter(targets['ko_fc'], targets['n_drugs'], s=8 + targets['total_score'] * 1.6,
                c=targets['total_score'], cmap='viridis', alpha=0.8, linewidths=0.2,
                edgecolors='white')
for _, r in top.head(6).iterrows():
    axB.annotate(r['gene'], (r['ko_fc'], r['n_drugs']), fontsize=5, style='italic',
                 xytext=(3, 2), textcoords='offset points')
cb = plt.colorbar(s, ax=axB, fraction=0.04, pad=0.02); cb.set_label('Priority score', fontsize=5.5)
cb.ax.tick_params(labelsize=5, width=0.4, length=1.5); cb.outline.set_linewidth(0.4)
axB.set_xlabel('|log$_2$ FC| (KO vs WT)'); axB.set_ylabel('Known drugs (n)'); clean(axB)
axB.set_title('Druggability vs effect size', pad=3); panel_label(axB, 'B', x=-0.22)

# C - drugs per gene
axC = fig.add_subplot(gs[0, 2])
if len(drugged):
    dd = drugged.nlargest(14, 'n_drugs')
    axC.barh(range(len(dd)), dd['n_drugs'], height=0.72, color='#3B75AF', linewidth=0)
    axC.set_yticks(range(len(dd))); axC.set_yticklabels(dd['gene'], fontsize=5.5, style='italic')
    axC.invert_yaxis(); axC.set_xlabel('Known drugs (n)')
clean(axC); axC.set_title('Drug coverage (DGIdb)', pad=3); panel_label(axC, 'C', x=-0.34)

# D - interaction types
axD = fig.add_subplot(gs[1, 0])
if drug_int is not None and len(drug_int):
    it = drug_int['interaction_type'].replace('Unknown', 'unspecified').value_counts().head(9)
    axD.barh(range(len(it)), it.values, height=0.72, color='#519E3E', linewidth=0)
    axD.set_yticks(range(len(it))); axD.set_yticklabels(it.index, fontsize=5.5)
    axD.invert_yaxis(); axD.set_xlabel('Interactions (n)')
clean(axD); axD.set_title('Drug-gene interaction types', pad=3); panel_label(axD, 'D', x=-0.34)

# E - evidence sources
axE = fig.add_subplot(gs[1, 1])
if drug_int is not None and len(drug_int):
    srcs = (drug_int['source'].fillna('').str.split(',').explode().str.strip()
            .replace('', np.nan).dropna().value_counts().head(9))
    axE.barh(range(len(srcs)), srcs.values, height=0.72, color='#8D69B8', linewidth=0)
    axE.set_yticks(range(len(srcs))); axE.set_yticklabels(srcs.index, fontsize=5.5)
    axE.invert_yaxis(); axE.set_xlabel('Interactions (n)')
clean(axE); axE.set_title('Evidence sources', pad=3); panel_label(axE, 'E', x=-0.34)

# F - funnel
axF = fig.add_subplot(gs[1, 2])
stages = ['Genes\ntested', 'Compensatory', 'With known\ndrugs']
vals = [len(dose_df), int((dose_df['pattern'] == 'Compensatory').sum()),
        int(drugged['gene'].nunique()) if len(drugged) else 0]
axF.bar(range(3), vals, width=0.62, color=['#BFBFBF', '#C53A32', '#3B75AF'], linewidth=0)
for i, v in enumerate(vals):
    axF.text(i, v * 1.15 if v else 0.5, f'{v:,}', ha='center', fontsize=6, fontweight='bold')
axF.set_yscale('log'); axF.set_xticks(range(3)); axF.set_xticklabels(stages, fontsize=5.5)
axF.set_ylabel('Genes (log scale)'); clean(axF)
axF.set_title('Target funnel', pad=3); panel_label(axF, 'F', x=-0.22)

save_figure(fig, 'Figure6_therapeutic_targets', 'main',
    "Figure 6. Prioritisation of druggable compensatory targets. "
    "(A) The 15 highest-scoring targets by the composite priority score (compensatory status, "
    "druggability, Fezf2-network membership and effect size). "
    "(B) Number of known drugs against effect size; point size and colour encode priority score. "
    "(C) Genes with the broadest existing drug coverage in DGIdb. "
    "(D) Distribution of drug-gene interaction types. "
    "(E) Evidence sources supporting the retrieved interactions. "
    "(F) Funnel from all tested genes, to compensatory genes, to those with an existing drug. "
    "DGIdb is keyed on human symbols; mouse genes were matched to human orthologs by symbol.")

  saved Figure6_therapeutic_targets


---
# Supplementary Figures

## Figure S1 — Quality control

In [12]:
qc_cols = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo']
qc_labels = ['Genes per cell', 'UMI counts per cell', 'Mitochondrial (%)', 'Ribosomal (%)']
samples = list(adata.obs['sample_id'].cat.categories) if hasattr(adata.obs['sample_id'], 'cat') \
    else sorted(adata.obs['sample_id'].unique())

fig, axes = plt.subplots(4, 1, figsize=(W_DOUBLE, 0.95 * W_DOUBLE), sharex=True)
for ax, col, lab, letter in zip(axes, qc_cols, qc_labels, 'ABCD'):
    data = [adata.obs.loc[adata.obs['sample_id'] == s, col].values for s in samples]
    parts = ax.violinplot(data, showextrema=False, widths=0.85)
    for pc in parts['bodies']:
        pc.set_facecolor('#3B75AF'); pc.set_alpha(0.75); pc.set_linewidth(0)
    ax.scatter(range(1, len(data) + 1), [np.median(d) for d in data], s=3,
               color='#222222', zorder=3, linewidths=0)
    if col == 'total_counts':
        ax.set_yscale('log')
    ax.set_ylabel(lab); clean(ax); panel_label(ax, letter, x=-0.06)
axes[-1].set_xticks(range(1, len(samples) + 1))
axes[-1].set_xticklabels(samples, rotation=90, fontsize=5)
axes[-1].set_xlabel('Sample')
axes[0].set_title('Per-sample quality control metrics (post-filtering)', pad=4)

save_figure(fig, 'FigureS1_quality_control', 'supp',
    "Figure S1. Per-sample quality control after filtering. Violins show the distribution across "
    "cells for each retained sample; black dots mark medians. (A) Genes detected per cell. "
    "(B) Total UMI counts per cell (log scale). (C) Mitochondrial read percentage, a proxy for "
    "cell stress or lysis. (D) Ribosomal read percentage.")

  saved FigureS1_quality_control


## Figure S2 — Doublet detection

In [13]:
doub_cols = [c for c in ['doublet_score', 'scrublet_score'] if c in adata.obs]
fig, axes = plt.subplots(1, 3, figsize=(W_DOUBLE, 0.26 * W_DOUBLE))

for i, c in enumerate(doub_cols[:2]):
    axes[i].hist(adata.obs[c].dropna(), bins=60, color='#3B75AF', linewidth=0)
    axes[i].set_xlabel(c.replace('_', ' ').capitalize()); axes[i].set_ylabel('Cells')
    axes[i].set_yscale('log'); clean(axes[i]); panel_label(axes[i], 'AB'[i], x=-0.2)
    axes[i].set_title(c.split('_')[0].capitalize() + ' score', pad=3)

ax = axes[2]
if len(doub_cols) == 2:
    ax.scatter(adata.obs[doub_cols[0]], adata.obs[doub_cols[1]], s=1, alpha=0.25,
               c='#666666', linewidths=0, rasterized=True)
    r = adata.obs[doub_cols].corr().iloc[0, 1]
    ax.text(0.04, 0.95, f'r = {r:.2f}', transform=ax.transAxes, va='top', fontsize=6)
    ax.set_xlabel('Solo score'); ax.set_ylabel('Scrublet score')
clean(ax); panel_label(ax, 'C', x=-0.2); ax.set_title('Method concordance', pad=3)
fig.tight_layout()

save_figure(fig, 'FigureS2_doublet_detection', 'supp',
    "Figure S2. Doublet detection. (A) Distribution of Solo doublet scores across retained cells "
    "(log-scaled y-axis). (B) Distribution of Scrublet doublet scores. (C) Concordance between the "
    "two methods; predicted doublets were removed before downstream analysis.")

  saved FigureS2_doublet_detection


## Figure S3 — Batch integration benchmark

In [14]:
fig = plt.figure(figsize=(W_DOUBLE, 0.32 * W_DOUBLE))
gs = fig.add_gridspec(1, 3, wspace=0.35)

sample_cmap = plt.get_cmap('tab20', len(samples))
smap = {s: sample_cmap(i) for i, s in enumerate(samples)}

axA = fig.add_subplot(gs[0, 0])
scatter_categorical(axA, UMAP, adata.obs['sample_id'].values, smap, order=samples, size=0.9)
axA.set_title('UMAP (Harmony) by sample', pad=2); panel_label(axA, 'A', x=-0.02)

axB = fig.add_subplot(gs[0, 1])
if 'X_umap_scVI' in adata.obsm:
    scatter_categorical(axB, adata.obsm['X_umap_scVI'], adata.obs['sample_id'].values,
                        smap, order=samples, size=0.9)
axB.set_title('UMAP (scVI) by sample', pad=2); panel_label(axB, 'B', x=-0.02)

axC = fig.add_subplot(gs[0, 2])
if integ_metrics is not None:
    m = integ_metrics.copy()
    x = np.arange(len(m)); w = 0.36
    axC.bar(x - w / 2, m['Harmony'].astype(float), width=w, color='#3B75AF', label='Harmony', linewidth=0)
    axC.bar(x + w / 2, m['scVI'].astype(float), width=w, color='#EF8636', label='scVI', linewidth=0)
    axC.set_xticks(x)
    axC.set_xticklabels([t.replace(' (', '\n(') for t in m['Metric']], fontsize=5, rotation=0)
    axC.axhline(0, lw=0.4, color='#999999')
    axC.set_ylabel('Score'); axC.legend(fontsize=5.5)
clean(axC); axC.set_title('Integration benchmark', pad=3); panel_label(axC, 'C', x=-0.2)

save_figure(fig, 'FigureS3_batch_integration', 'supp',
    "Figure S3. Batch integration benchmark. (A) UMAP computed on the Harmony-corrected space, "
    "coloured by sample of origin; even mixing of colours indicates the batch effect has been "
    "removed. (B) The equivalent embedding from the scVI latent space. (C) Quantitative comparison "
    "of the two integration methods.")

  saved FigureS3_batch_integration


## Figure S4 — Clustering and marker-based annotation

In [15]:
fig = plt.figure(figsize=(W_DOUBLE, 0.40 * W_DOUBLE))
gs = fig.add_gridspec(1, 3, wspace=0.4, width_ratios=[1, 1, 1.2])

leiden_key = 'leiden_r0.8' if 'leiden_r0.8' in adata.obs else 'leiden'
clusters = sorted(adata.obs[leiden_key].astype(str).unique(), key=lambda s: int(s))
lc = plt.get_cmap('tab20', len(clusters))

axA = fig.add_subplot(gs[0, 0])
scatter_categorical(axA, UMAP, adata.obs[leiden_key].astype(str).values,
                    {c: lc(i) for i, c in enumerate(clusters)}, order=clusters, size=1.0)
for c in clusters:
    m = (adata.obs[leiden_key].astype(str) == c).values
    axA.text(UMAP[m, 0].mean(), UMAP[m, 1].mean(), c, fontsize=4.5, ha='center', va='center',
             fontweight='bold')
axA.set_title(f'Leiden clusters (n={len(clusters)})', pad=2); panel_label(axA, 'A', x=-0.02)

axB = fig.add_subplot(gs[0, 1])
sizes = adata.obs[leiden_key].astype(str).value_counts().reindex(clusters)
axB.bar(range(len(sizes)), sizes.values, width=0.75,
        color=[lc(i) for i in range(len(clusters))], linewidth=0)
axB.set_xticks(range(len(sizes))); axB.set_xticklabels(sizes.index, fontsize=5, rotation=90)
axB.set_ylabel('Cells'); axB.set_xlabel('Leiden cluster'); clean(axB)
axB.set_title('Cluster sizes', pad=3); panel_label(axB, 'B', x=-0.2)

axC = fig.add_subplot(gs[0, 2])
ctab = pd.crosstab(adata.obs[leiden_key].astype(str), adata.obs['cell_type'],
                   normalize='index') * 100
ctab = ctab.reindex(index=clusters, columns=CELLTYPE_ORDER, fill_value=0)
im = axC.imshow(ctab.values, aspect='auto', cmap='Blues', vmin=0, vmax=100)
axC.set_xticks(range(len(CELLTYPE_ORDER)))
axC.set_xticklabels(CELLTYPE_ORDER, rotation=90, fontsize=5)
axC.set_yticks(range(len(clusters))); axC.set_yticklabels(clusters, fontsize=5)
axC.set_ylabel('Leiden cluster'); axC.tick_params(length=0)
cb = plt.colorbar(im, ax=axC, fraction=0.03, pad=0.02); cb.set_label('% of cluster', fontsize=5.5)
cb.ax.tick_params(labelsize=5, width=0.4, length=1.5); cb.outline.set_linewidth(0.4)
axC.set_title('Cluster to cell-type mapping', pad=3); panel_label(axC, 'C', x=-0.14)

save_figure(fig, 'FigureS4_clustering', 'supp',
    "Figure S4. Unsupervised clustering and its mapping onto annotated cell types. "
    "(A) Leiden clusters (resolution 0.8) on the UMAP, labelled at their centroids. "
    "(B) Number of cells per cluster. (C) Heat map showing, for each Leiden cluster, the "
    "percentage of its cells assigned to each annotated cell type.")

  saved FigureS4_clustering


## Figure S5 — Marker score maps

In [16]:
score_cols = [c for c in adata.obs.columns if c.startswith('score_')]
n = len(score_cols)
ncol = 5
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(W_DOUBLE, W_DOUBLE * 0.22 * nrow))
axes = np.atleast_1d(axes).ravel()
for ax, c in zip(axes, score_cols):
    scatter_continuous(ax, UMAP, adata.obs[c].values, cmap='viridis', size=0.8, cbar=False)
    ax.set_title(c.replace('score_', '').replace('_', ' '), fontsize=5.5, pad=1)
for ax in axes[n:]:
    ax.axis('off')
fig.suptitle('Marker-gene signature scores', fontsize=7, y=1.0)

save_figure(fig, 'FigureS5_marker_scores', 'supp',
    "Figure S5. Marker-gene signature scores for each candidate cell type projected onto the UMAP. "
    "Each panel scores every cell against a literature-defined marker panel; these scores drove the "
    "automated annotation that was then manually curated.")

  saved FigureS5_marker_scores


## Figure S6 — Composition by genotype

In [17]:
fig = plt.figure(figsize=(W_DOUBLE, 0.42 * W_DOUBLE))
gs = fig.add_gridspec(1, 2, wspace=0.3, width_ratios=[1.35, 1])

axA = fig.add_subplot(gs[0, 0])
matched = ['E13', 'E15', 'P1']
groups, labels = [], []
for tp in matched:
    for g in GENOTYPE_ORDER:
        m = ((adata.obs['timepoint'] == tp) & (adata.obs['genotype'] == g)).values
        if m.sum() > 0:
            groups.append(m); labels.append(f'{tp}\n{g}')
comp = pd.DataFrame(
    [pd.Series(adata.obs['cell_type'].values[m]).value_counts(normalize=True).reindex(
        CELLTYPE_ORDER, fill_value=0) * 100 for m in groups], index=labels)
bottom = np.zeros(len(comp))
for ct in CELLTYPE_ORDER:
    axA.bar(range(len(comp)), comp[ct].values, bottom=bottom, width=0.78,
            color=CT_COLORS[ct], linewidth=0, label=ct)
    bottom += comp[ct].values
axA.set_xticks(range(len(comp))); axA.set_xticklabels(comp.index, fontsize=5.5)
axA.set_ylabel('Composition (%)'); axA.set_ylim(0, 100); clean(axA)
axA.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=5)
axA.set_title('Cell type composition by stage and genotype', pad=3)
panel_label(axA, 'A', x=-0.09)

axB = fig.add_subplot(gs[0, 1])
p1 = adata.obs['timepoint'] == 'P1'
p1c = pd.crosstab(adata.obs['cell_type'][p1], adata.obs['genotype'][p1], normalize='columns') * 100
p1c = p1c.reindex(index=CELLTYPE_ORDER, columns=GENOTYPE_ORDER, fill_value=0)
lfc = np.log2((p1c['KO'] + 0.1) / (p1c['WT'] + 0.1))
cols = ['#C53A32' if v > 0 else '#3B75AF' for v in lfc]
axB.barh(range(len(lfc)), lfc.values, height=0.72, color=cols, linewidth=0)
axB.set_yticks(range(len(lfc))); axB.set_yticklabels(lfc.index, fontsize=5.5)
axB.invert_yaxis(); axB.axvline(0, lw=0.5, color='#333333')
axB.set_xlabel('log$_2$ (KO / WT) proportion'); clean(axB)
axB.set_title('Compositional shift at P1', pad=3); panel_label(axB, 'B', x=-0.5)

save_figure(fig, 'FigureS6_composition_by_genotype', 'supp',
    "Figure S6. Cellular composition across genotypes. (A) Stacked cell-type composition at the "
    "three matched stages; note that no WT cells exist at E13 or E15. (B) Log2 ratio of KO to WT "
    "cell-type proportions at P1; red indicates over-representation in KO.")

  saved FigureS6_composition_by_genotype


## Figure S7 — Sex-dimorphic response

In [18]:
fig = plt.figure(figsize=(W_DOUBLE, 0.30 * W_DOUBLE))
gs = fig.add_gridspec(1, 3, wspace=0.4)

axA = fig.add_subplot(gs[0, 0])
if sex_de is not None:
    v = sex_de.dropna(subset=['logfoldchanges', 'pvals_adj']).copy()
    v['nlp'] = -np.log10(v['pvals_adj'].clip(lower=1e-300))
    sig = (v['pvals_adj'] < 0.05) & (v['logfoldchanges'].abs() > 0.5)
    axA.scatter(v.loc[~sig, 'logfoldchanges'], v.loc[~sig, 'nlp'], s=1.5, c='#CCCCCC',
                linewidths=0, rasterized=True)
    axA.scatter(v.loc[sig & (v['logfoldchanges'] > 0), 'logfoldchanges'],
                v.loc[sig & (v['logfoldchanges'] > 0), 'nlp'], s=2, c='#D95F9A',
                linewidths=0, rasterized=True, label='Female-biased')
    axA.scatter(v.loc[sig & (v['logfoldchanges'] < 0), 'logfoldchanges'],
                v.loc[sig & (v['logfoldchanges'] < 0), 'nlp'], s=2, c='#3B75AF',
                linewidths=0, rasterized=True, label='Male-biased')
    axA.axhline(-np.log10(0.05), ls='--', lw=0.4, color='#555555')
    for _, r in v.loc[sig].nlargest(6, 'nlp').iterrows():
        axA.annotate(r['names'], (r['logfoldchanges'], r['nlp']), fontsize=4.5, style='italic',
                     xytext=(2, 1), textcoords='offset points')
    axA.set_xlabel('log$_2$ FC (Female vs Male)'); axA.set_ylabel('-log$_{10}$ FDR')
    axA.legend(fontsize=5, markerscale=2)
clean(axA); axA.set_title('Sex-dimorphic DE in Het, P1', pad=3); panel_label(axA, 'A', x=-0.2)

axB = fig.add_subplot(gs[0, 1])
if sex_de is not None:
    n_f = int((sig & (v['logfoldchanges'] > 0)).sum()); n_m = int((sig & (v['logfoldchanges'] < 0)).sum())
    axB.bar([0, 1], [n_f, n_m], width=0.6, color=['#D95F9A', '#3B75AF'], linewidth=0)
    for i, val in enumerate([n_f, n_m]):
        axB.text(i, val + 4, str(val), ha='center', fontsize=6)
    axB.set_xticks([0, 1]); axB.set_xticklabels(['Female-\nbiased', 'Male-\nbiased'])
    axB.set_ylabel('Significant genes (n)'); axB.set_ylim(0, max(n_f, n_m) * 1.2)
clean(axB); axB.set_title('Directional bias', pad=3); panel_label(axB, 'B', x=-0.25)

axC = fig.add_subplot(gs[0, 2])
if sex_de is not None and sig.any():
    topg = v.loc[sig].reindex(v.loc[sig, 'logfoldchanges'].abs().nlargest(14).index)
    cols = ['#D95F9A' if x > 0 else '#3B75AF' for x in topg['logfoldchanges']]
    axC.barh(range(len(topg)), topg['logfoldchanges'], height=0.72, color=cols, linewidth=0)
    axC.set_yticks(range(len(topg))); axC.set_yticklabels(topg['names'], fontsize=5, style='italic')
    axC.invert_yaxis(); axC.axvline(0, lw=0.5, color='#333333')
    axC.set_xlabel('log$_2$ FC (Female vs Male)')
clean(axC); axC.set_title('Strongest sex-biased genes', pad=3); panel_label(axC, 'C', x=-0.4)

save_figure(fig, 'FigureS7_sex_dimorphism', 'supp',
    "Figure S7. Sex-dimorphic transcriptional response in Fezf2 heterozygotes at P1. "
    "(A) Volcano plot of female-versus-male differential expression (FDR < 0.05, |log2 FC| > 0.5). "
    "(B) Number of significant female- and male-biased genes. (C) Genes with the largest "
    "sex-biased fold changes.")

  saved FigureS7_sex_dimorphism


## Figure S8 — Extended dose-response diagnostics

In [19]:
fig = plt.figure(figsize=(W_DOUBLE, 0.30 * W_DOUBLE))
gs = fig.add_gridspec(1, 3, wspace=0.4)

axA = fig.add_subplot(gs[0, 0])
axA.hist(dose_df['additivity_deviation'], bins=60, color='#8D69B8', linewidth=0)
axA.axvline(0, lw=0.5, color='#333333')
axA.set_xlabel('Additivity deviation (Het - expected)'); axA.set_ylabel('Genes')
clean(axA); axA.set_title('Deviation from additivity', pad=3); panel_label(axA, 'A', x=-0.2)

axB = fig.add_subplot(gs[0, 1])
axB.scatter(dose_df['WT_mean'], dose_df['KO_mean'], s=2, alpha=0.4, c='#999999',
            linewidths=0, rasterized=True)
comp_m = dose_df['pattern'] == 'Compensatory'
axB.scatter(dose_df.loc[comp_m, 'WT_mean'], dose_df.loc[comp_m, 'KO_mean'], s=2.5, alpha=0.8,
            c=PATTERN_COLORS['Compensatory'], linewidths=0, rasterized=True, label='Compensatory')
mx = float(np.nanmax(dose_df[['WT_mean', 'KO_mean']].values))
axB.plot([0, mx], [0, mx], ls='--', lw=0.5, color='#333333')
axB.set_xlabel('WT mean (log-norm)'); axB.set_ylabel('KO mean (log-norm)')
axB.legend(fontsize=5.5, markerscale=2.5); clean(axB)
axB.set_title('WT vs KO expression', pad=3); panel_label(axB, 'B', x=-0.2)

axC = fig.add_subplot(gs[0, 2])
axC.scatter(dose_df['dose_correlation'], dose_df['KO_vs_WT_fc'], s=2, alpha=0.45,
            c=[PATTERN_COLORS[p] for p in dose_df['pattern']], linewidths=0, rasterized=True)
axC.axhline(0, lw=0.4, color='#999999'); axC.axvline(0, lw=0.4, color='#999999')
axC.set_xlabel('Spearman r with dosage'); axC.set_ylabel('log$_2$ FC (KO vs WT)')
axC.legend([Patch(facecolor=PATTERN_COLORS[p]) for p in PATTERN_COLORS if p in set(dose_df['pattern'])],
           [p for p in PATTERN_COLORS if p in set(dose_df['pattern'])], fontsize=5, loc='upper right')
clean(axC); axC.set_title('Correlation vs effect size', pad=3); panel_label(axC, 'C', x=-0.2)

save_figure(fig, 'FigureS8_dose_response_diagnostics', 'supp',
    "Figure S8. Diagnostics for the dose-response classification. (A) Distribution of the "
    "additivity deviation (observed Het minus the midpoint of WT and KO); values near zero "
    "indicate an additive, gene-dosage-like response. (B) WT versus KO pseudobulk expression, "
    "with compensatory genes highlighted above the diagonal. (C) Dosage correlation against KO "
    "effect size, coloured by assigned class.")

  saved FigureS8_dose_response_diagnostics


## Figure S9 — Transcription-factor networks

In [20]:
sizes = pd.Series({tf: len(d) for tf, d in tf_networks.items()}).sort_values(ascending=False)
top_tfs = [t for t in sizes.index if len(tf_networks[t]) >= 3][:6]

fig = plt.figure(figsize=(W_DOUBLE, 0.30 * W_DOUBLE))
gs = fig.add_gridspec(1, 2, wspace=0.32, width_ratios=[1, 1.15])

axA = fig.add_subplot(gs[0, 0])
for tf in top_tfs:
    d = tf_networks[tf]
    axA.hist(d['correlation'], bins=30, histtype='step', lw=1, label=tf, density=True)
axA.set_xlabel('Correlation with TF'); axA.set_ylabel('Density')
axA.legend(fontsize=5, ncol=2); clean(axA)
axA.set_title('Target correlation distributions', pad=3); panel_label(axA, 'A', x=-0.2)

axB = fig.add_subplot(gs[0, 1])
tf_list = list(sizes.index)
gene_sets = {tf: set(tf_networks[tf]['gene']) for tf in tf_list if 'gene' in tf_networks[tf]}
tf_list = [t for t in tf_list if t in gene_sets]
J = np.zeros((len(tf_list), len(tf_list)))
for i, a_ in enumerate(tf_list):
    for j, b_ in enumerate(tf_list):
        u = gene_sets[a_] | gene_sets[b_]
        J[i, j] = len(gene_sets[a_] & gene_sets[b_]) / len(u) if u else 0
im = axB.imshow(J, cmap='magma', vmin=0, vmax=1)
axB.set_xticks(range(len(tf_list))); axB.set_xticklabels(tf_list, rotation=90, fontsize=5, style='italic')
axB.set_yticks(range(len(tf_list))); axB.set_yticklabels(tf_list, fontsize=5, style='italic')
axB.tick_params(length=0)
cb = plt.colorbar(im, ax=axB, fraction=0.04, pad=0.02)
cb.set_label('Jaccard overlap', fontsize=5.5)
cb.ax.tick_params(labelsize=5, width=0.4, length=1.5); cb.outline.set_linewidth(0.4)
axB.set_title('Target-set overlap between TFs', pad=3); panel_label(axB, 'B', x=-0.16)

save_figure(fig, 'FigureS9_tf_networks', 'supp',
    "Figure S9. Transcription-factor correlation networks at E13.5 (WT). (A) Distribution of "
    "target correlations for the six TFs with the largest networks. (B) Jaccard overlap between "
    "the target sets of every TF, showing which regulators share downstream genes.")

  saved FigureS9_tf_networks


## Figure S10 — Drug landscape

In [21]:
fig = plt.figure(figsize=(W_DOUBLE, 0.30 * W_DOUBLE))
gs = fig.add_gridspec(1, 3, wspace=0.42)

axA = fig.add_subplot(gs[0, 0])
if drug_int is not None and len(drug_int):
    per_gene = drug_int.groupby('gene').size().sort_values(ascending=False)
    axA.hist(per_gene.values, bins=20, color='#3B75AF', linewidth=0)
    axA.set_xlabel('Drugs per gene'); axA.set_ylabel('Genes')
clean(axA); axA.set_title('Drug coverage distribution', pad=3); panel_label(axA, 'A', x=-0.2)

axB = fig.add_subplot(gs[0, 1])
if drug_int is not None and len(drug_int):
    top_drugs = drug_int.groupby('drug')['gene'].nunique().sort_values(ascending=False).head(12)
    axB.barh(range(len(top_drugs)), top_drugs.values, height=0.72, color='#EF8636', linewidth=0)
    axB.set_yticks(range(len(top_drugs)))
    axB.set_yticklabels([d.title()[:22] for d in top_drugs.index], fontsize=5)
    axB.invert_yaxis(); axB.set_xlabel('Target genes hit (n)')
clean(axB); axB.set_title('Most promiscuous drugs', pad=3); panel_label(axB, 'B', x=-0.42)

axC = fig.add_subplot(gs[0, 2])
if targets is not None and 'ml_score' in targets:
    axC.scatter(targets['total_score'], targets['ml_score'], s=4, alpha=0.6, c='#519E3E',
                linewidths=0, rasterized=True)
    r = targets[['total_score', 'ml_score']].corr().iloc[0, 1]
    axC.text(0.04, 0.95, f'r = {r:.3f}', transform=axC.transAxes, va='top', fontsize=6)
    axC.set_xlabel('Composite priority score'); axC.set_ylabel('Gradient-boosting score')
clean(axC); axC.set_title('Score concordance', pad=3); panel_label(axC, 'C', x=-0.2)

save_figure(fig, 'FigureS10_drug_landscape', 'supp',
    "Figure S10. Drug-gene interaction landscape for the compensatory gene set. (A) Distribution "
    "of the number of known drugs per druggable gene. (B) Drugs hitting the largest number of "
    "candidate targets. (C) Agreement between the composite priority score and the "
    "gradient-boosting model ranking.")

  saved FigureS10_drug_landscape


---
# Main Tables

In [22]:
# ---- Table 1: sample cohort
t1 = (adata.obs.groupby('sample_id', observed=True)
      .agg(timepoint=('timepoint', 'first'), genotype=('genotype', 'first'),
           sex=('sex', 'first'), n_cells=('sample_id', 'size'),
           median_genes=('n_genes_by_counts', 'median'),
           median_umis=('total_counts', 'median'),
           median_pct_mito=('pct_counts_mt', 'median'))
      .reset_index())
t1['timepoint'] = pd.Categorical(t1['timepoint'], categories=present_tp, ordered=True)
t1 = t1.sort_values(['timepoint', 'genotype']).round(1)
save_table(t1, 'Table1_sample_cohort', 'main',
    "Table 1. Sample cohort. One row per scRNA-seq library, giving developmental stage, Fezf2 "
    "genotype, sex, number of cells retained after quality control, and median per-cell quality "
    "metrics.")

# ---- Table 2: cell type summary
n_tot = adata.n_obs
ct = adata.obs.groupby('cell_type', observed=True).size().reindex(CELLTYPE_ORDER, fill_value=0)
geno_ct = pd.crosstab(adata.obs['cell_type'], adata.obs['genotype']).reindex(
    index=CELLTYPE_ORDER, columns=GENOTYPE_ORDER, fill_value=0)
markers_used = {k: [g for g in v if g in adata.var_names] for k, v in MARKERS.items()}
t2 = pd.DataFrame({
    'cell_type': CELLTYPE_ORDER,
    'n_cells': ct.values,
    'pct_of_dataset': (ct.values / n_tot * 100).round(2),
    'n_WT': geno_ct['WT'].values, 'n_Het': geno_ct['Het'].values, 'n_KO': geno_ct['KO'].values,
    'marker_genes': [', '.join(markers_used.get(c, [])) for c in CELLTYPE_ORDER],
})
save_table(t2, 'Table2_celltype_summary', 'main',
    "Table 2. Annotated cell types. Cell counts overall and per genotype, together with the "
    "canonical marker genes supporting each annotation.")

# ---- Table 3: dose-response summary
rows = []
for p in pat_order:
    d = dose_df[dose_df['pattern'] == p]
    top = d.loc[d['KO_vs_WT_fc'].abs().nlargest(10).index, 'gene'].tolist()
    rows.append({'pattern': p, 'n_genes': len(d),
                 'pct_of_tested': round(100 * len(d) / len(dose_df), 1),
                 'median_KO_vs_WT_log2FC': round(float(d['KO_vs_WT_fc'].median()), 3),
                 'median_dose_correlation': round(float(d['dose_correlation'].median()), 3),
                 'top10_genes': ', '.join(top)})
t3 = pd.DataFrame(rows)
save_table(t3, 'Table3_dose_response_summary', 'main',
    "Table 3. Dose-response classes at P1. For each class: the number and percentage of genes, "
    "the median KO-versus-WT effect size, the median Spearman correlation with Fezf2 dosage, and "
    "the ten genes with the largest absolute effect.")

# ---- Table 4: prioritised therapeutic targets
t4 = targets.nlargest(25, 'total_score').copy()
if drug_int is not None and len(drug_int):
    dmap = drug_int.groupby('gene')['drug'].apply(lambda s: ', '.join(sorted(set(s))[:6]))
    t4['example_drugs'] = t4['gene'].map(dmap).fillna('-')
pat_map = dose_df.set_index('gene')['pattern'].to_dict()
t4['dose_response_class'] = t4['gene'].map(pat_map)
t4 = t4[['gene', 'dose_response_class', 'ko_fc', 'n_drugs', 'is_fezf2_target',
         'total_score', 'ml_score'] + (['example_drugs'] if 'example_drugs' in t4 else [])].round(3)
save_table(t4, 'Table4_therapeutic_targets', 'main',
    "Table 4. Top 25 prioritised therapeutic targets, ranked by composite priority score, with "
    "dose-response class, KO effect size, number of known drugs in DGIdb, Fezf2-network "
    "membership, and example drugs.")

  saved Table1_sample_cohort  (20 rows)
  saved Table2_celltype_summary  (14 rows)
  saved Table3_dose_response_summary  (5 rows)
  saved Table4_therapeutic_targets  (25 rows)


---
# Supplementary Tables

In [23]:
# S1 - per-sample QC
save_table(t1, 'TableS1_qc_per_sample', 'supp',
    "Table S1. Per-sample quality-control metrics after filtering (expanded form of Table 1).")

# S2 - cluster markers
if cluster_mark is not None:
    save_table(cluster_mark, 'TableS2_cluster_markers', 'supp',
        "Table S2. Differentially expressed marker genes for every Leiden cluster "
        "(t-test with overestimated variance; Benjamini-Hochberg FDR).")

# S3 - composition
comp_tab = (adata.obs.groupby(['timepoint', 'genotype', 'cell_type'], observed=True)
            .size().reset_index(name='n_cells'))
comp_tab['pct_of_group'] = (comp_tab.groupby(['timepoint', 'genotype'], observed=True)['n_cells']
                            .transform(lambda s: 100 * s / s.sum()).round(2))
save_table(comp_tab, 'TableS3_celltype_composition', 'supp',
    "Table S3. Cell-type composition for every stage-by-genotype group, as absolute counts and as "
    "a percentage of that group.")

# S4 - full dose-response
save_table(dose_df, 'TableS4_dose_response_full', 'supp',
    "Table S4. Complete dose-response results at P1 for all tested genes: per-genotype pseudobulk "
    "means, log2 fold changes, Spearman correlation with Fezf2 dosage, additivity deviation, and "
    "assigned class.")

# S5 - compensatory genes
comp_full = dose_df[dose_df['pattern'] == 'Compensatory'].sort_values('KO_vs_WT_fc', ascending=False)
save_table(comp_full, 'TableS5_compensatory_genes', 'supp',
    "Table S5. All compensatory genes (up-regulated as Fezf2 dosage falls), ordered by KO-versus-WT "
    "effect size. These form the input to therapeutic prioritisation.")

# S6 - sex-dimorphic
if sex_de is not None:
    save_table(sex_de, 'TableS6_sex_dimorphic_genes', 'supp',
        "Table S6. Female-versus-male differential expression in Fezf2 heterozygotes at P1.")

# S7 - DE WT vs KO
if de_wt_ko is not None:
    save_table(de_wt_ko, 'TableS7_de_wt_vs_ko_p1', 'supp',
        "Table S7. Differential expression between WT and KO at P1 (t-test with overestimated "
        "variance; Benjamini-Hochberg FDR).")

# S8 - TF networks
tf_long = pd.concat(
    [d.assign(TF=tf) for tf, d in tf_networks.items() if len(d)],
    ignore_index=True) if tf_networks else pd.DataFrame()
if len(tf_long):
    tf_long = tf_long[['TF'] + [c for c in tf_long.columns if c != 'TF']]
save_table(tf_long, 'TableS8_tf_networks', 'supp',
    "Table S8. Correlation-based regulatory networks for all cortical transcription factors at "
    "E13.5 (WT): every TF-target pair passing |r| > 0.3. Associative, not causal.")

# S9 - drug interactions
if drug_int is not None:
    save_table(drug_int, 'TableS9_drug_gene_interactions', 'supp',
        "Table S9. All drug-gene interactions retrieved from DGIdb for the compensatory gene set, "
        "with interaction type and supporting source database.")

# S10 - full target ranking
if targets is not None:
    tt = targets.copy()
    tt['dose_response_class'] = tt['gene'].map(pat_map)
    save_table(tt, 'TableS10_prioritised_targets_full', 'supp',
        "Table S10. Complete therapeutic-target scoring table for every compensatory gene, with "
        "all scoring components and the gradient-boosting rank.")

  saved TableS1_qc_per_sample  (20 rows)
  saved TableS2_cluster_markers  (2,410 rows)
  saved TableS3_celltype_composition  (209 rows)
  saved TableS4_dose_response_full  (3,000 rows)
  saved TableS5_compensatory_genes  (694 rows)
  saved TableS6_sex_dimorphic_genes  (3,000 rows)
  saved TableS7_de_wt_vs_ko_p1  (200 rows)
  saved TableS8_tf_networks  (804 rows)
  saved TableS9_drug_gene_interactions  (179 rows)
  saved TableS10_prioritised_targets_full  (694 rows)


---
# Index and captions

In [24]:
index_df = pd.DataFrame(MANIFEST)
order = {'main figure': 0, 'main table': 1, 'supp figure': 2, 'supp table': 3}
index_df['_o'] = index_df['type'].map(order)
index_df = index_df.sort_values(['_o', 'name']).drop(columns='_o')
index_df.to_csv(PUB / 'FIGURE_INDEX.csv', index=False)

lines = ["# Figure and Table Captions\n",
         "Auto-generated by `06_publication_figures.ipynb`. "
         "Figures are vector PDF (600 dpi, TrueType); PNG previews accompany each.\n"]
for section, key in [("## Main Figures", 'main figure'), ("## Main Tables", 'main table'),
                     ("## Supplementary Figures", 'supp figure'),
                     ("## Supplementary Tables", 'supp table')]:
    sub = index_df[index_df['type'] == key]
    if not len(sub):
        continue
    lines.append(f"\n{section}\n")
    for _, r in sub.iterrows():
        lines.append(f"**`{r['name']}`** — `{r['file']}`\n")
        lines.append(f"{r['caption']}\n")
(PUB / 'CAPTIONS.md').write_text('\n'.join(lines))

print(index_df[['type', 'name']].to_string(index=False))
print(f"\nWrote {PUB/'FIGURE_INDEX.csv'}")
print(f"Wrote {PUB/'CAPTIONS.md'}")

       type                               name
main figure          Figure1_single_cell_atlas
main figure           Figure2_fezf2_expression
main figure               Figure3_trajectories
main figure              Figure4_dose_response
main figure          Figure5_chromatin_and_grn
main figure        Figure6_therapeutic_targets
 main table               Table1_sample_cohort
 main table            Table2_celltype_summary
 main table       Table3_dose_response_summary
 main table         Table4_therapeutic_targets
supp figure           FigureS10_drug_landscape
supp figure           FigureS1_quality_control
supp figure         FigureS2_doublet_detection
supp figure         FigureS3_batch_integration
supp figure                FigureS4_clustering
supp figure             FigureS5_marker_scores
supp figure   FigureS6_composition_by_genotype
supp figure            FigureS7_sex_dimorphism
supp figure FigureS8_dose_response_diagnostics
supp figure               FigureS9_tf_networks
 supp table  

In [25]:
n_fig_main = sum(1 for m in MANIFEST if m['type'] == 'main figure')
n_fig_supp = sum(1 for m in MANIFEST if m['type'] == 'supp figure')
n_tab_main = sum(1 for m in MANIFEST if m['type'] == 'main table')
n_tab_supp = sum(1 for m in MANIFEST if m['type'] == 'supp table')

print("=" * 62)
print("PHASE 6 COMPLETE — PUBLICATION FIGURES & TABLES")
print("=" * 62)
print(f"  Main figures          : {n_fig_main}")
print(f"  Main tables           : {n_tab_main}")
print(f"  Supplementary figures : {n_fig_supp}")
print(f"  Supplementary tables  : {n_tab_supp}")
print(f"\n  Output: {PUB}")
print("\nCaveats carried into the manuscript:")
print("  - P1 is the only stage with all three genotypes (no WT at E13/E15).")
print("  - Dose-response statistics rest on three genotype pseudobulk means.")
print("  - TF networks are correlation-based: associative, not causal.")
print("  - DGIdb is keyed on human orthologs matched by gene symbol.")

PHASE 6 COMPLETE — PUBLICATION FIGURES & TABLES
  Main figures          : 6
  Main tables           : 4
  Supplementary figures : 10
  Supplementary tables  : 10

  Output: /Users/jubayer/Projects/single-cell/fezf2-multiomics/results/publication

Caveats carried into the manuscript:
  - P1 is the only stage with all three genotypes (no WT at E13/E15).
  - Dose-response statistics rest on three genotype pseudobulk means.
  - TF networks are correlation-based: associative, not causal.
  - DGIdb is keyed on human orthologs matched by gene symbol.
